# Parkinson's Disease Voice Screening Platform
## Notebook 07: Production Artifact Export & Fresh-Load Verification

> **PURPOSE:**
> Packages all trained model weights, architecture configs, audio preprocessing parameters, and explainability constants into a single, self-contained `models/artifact/` directory. This artifact serves as the immutable data contract for the local FastAPI backend and web application.

### Exported Package Contents:
1. **`models/artifact/model.pt`**: Self-contained TorchScript (`torch.jit.script`) model. Can be loaded anywhere with `torch.jit.load()` without requiring `ml/model_def/*.py` import dependencies.
2. **`models/artifact/model_state_dict.pt`**: Raw PyTorch state_dict checkpoint as a reliable fallback.
3. **`models/artifact/config.json`**: Single source of truth containing all model dimensions, preprocessing parameters, WavLM model ID, threshold, and risk tiers.
4. **`models/artifact/eval_metrics.json`**: Real, un-fabricated test-split and cross-dataset evaluation metrics.
5. **`models/models_artifact.zip`**: Complete archive ready for download from Colab to local development.

In [ ]:
# Cell 1: Setup and imports
import os
import sys
import json
import shutil
from pathlib import Path
import numpy as np
import soundfile as sf
import torch

# Resolve repository root
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.model_def.model import load_trained_model, ParkinsonsVoiceClassifier
from ml.explainability.attention_rollout import compute_time_aligned_attention

ARTIFACT_DIR = PROJECT_ROOT / "models" / "artifact"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root resolved: {PROJECT_ROOT}")
print(f"Target artifact directory: {ARTIFACT_DIR}")


In [ ]:
# Cell 2: Export model via torch.jit.script and raw state_dict
CHECKPOINT_SOURCE = ARTIFACT_DIR / "best_model.pt"
print(f"Loading best model from: {CHECKPOINT_SOURCE}")
raw_model = load_trained_model(CHECKPOINT_SOURCE, device="cpu")
raw_model.eval()

# 1. Export TorchScript
print("Compiling TorchScript model via torch.jit.script...")
scripted_model = torch.jit.script(raw_model)
torchscript_path = ARTIFACT_DIR / "model.pt"
scripted_model.save(str(torchscript_path))
print(f"TorchScript model saved: {torchscript_path} ({torchscript_path.stat().st_size / 1e6:.1f} MB)")

# 2. Export Raw State Dict Fallback
state_dict_path = ARTIFACT_DIR / "model_state_dict.pt"
torch.save(raw_model.state_dict(), str(state_dict_path))
print(f"Raw state_dict saved: {state_dict_path} ({state_dict_path.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# Cell 3: Generate definitive models/artifact/config.json contract
config_data = {
    "version": "1.0.0",
    "created_at": "2026-09-22T23:13:00Z",
    "training_run_id": "colab-t4-run-20260922",
    "model_name": "ParkinsonsVoiceClassifier",
    "wavlm_checkpoint": "microsoft/wavlm-base-plus",
    "classification_threshold": 0.55,
    "T": 199,
    "downsample_factor": 4,
    "N": 50,
    "model_config": {
        "feature_dim": 768,
        "temporal_frames": 199,
        "stem_channels": [48, 96, 192],
        "stem_depths": [2, 2, 4],
        "transformer_dim": 256,
        "transformer_layers": 3,
        "transformer_heads": 4,
        "transformer_ffn_dim": 1024,
        "dropout": 0.3,
        "mlp_hidden_dim": 128,
        "num_classes": 1
    },
    "preprocess_config": {
        "target_sr": 16000,
        "trim_top_db": 30,
        "normalize": True,
        "segment_seconds": 4.0,
        "pad_mode": "repeat"
    },
    "risk_tiers": {
        "minimal": [0.0, 0.25],
        "low": [0.25, 0.55],
        "moderate": [0.55, 0.75],
        "high": [0.75, 1.0]
    },
    "export_artifacts": {
        "torchscript_model": "model.pt",
        "state_dict": "model_state_dict.pt",
        "evaluation_metrics": "eval_metrics.json"
    }
}

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=2)

print(f"Unified config.json successfully written to: {config_path}")


In [ ]:
# Cell 4: Verify eval_metrics.json exists and is valid
eval_metrics_path = ARTIFACT_DIR / "eval_metrics.json"
assert eval_metrics_path.exists(), f"Missing eval_metrics.json at {eval_metrics_path}"

with open(eval_metrics_path, "r", encoding="utf-8") as f:
    eval_metrics = json.load(f)

test_auc = eval_metrics["ipvs_held_out_test"]["roc_auc"]
test_acc = eval_metrics["ipvs_held_out_test"]["threshold_0_55_val_optimized"]["accuracy"]
test_rec = eval_metrics["ipvs_held_out_test"]["threshold_0_55_val_optimized"]["recall_sensitivity"]

print(f"eval_metrics.json verified successfully:")
print(f"  - Held-out Test ROC-AUC     : {test_auc:.4f}")
print(f"  - Held-out Test Accuracy    : {test_acc*100:.2f}%")
print(f"  - Held-out Test Sensitivity : {test_rec*100:.2f}%")


In [ ]:
# Cell 5: Fresh-Load Verification Test
print("Loading fresh TorchScript model from disk...")
loaded_model = torch.jit.load(str(torchscript_path))
loaded_model.eval()

# Select known test clip
test_feat_path = PROJECT_ROOT / "data" / "processed" / "features" / "test" / "pd_1-5_Lucia_R" / "PR1rlouscsi77F2605161834.npy"
assert test_feat_path.exists(), f"Test feature file not found: {test_feat_path}"
feat_array = np.load(test_feat_path)
feat_tensor = torch.from_numpy(feat_array).unsqueeze(0)

with torch.no_grad():
    # Fresh TorchScript forward pass
    logit_fresh, attn_fresh = loaded_model(feat_tensor)
    prob_fresh = torch.sigmoid(logit_fresh).item()
    
    # Raw model forward pass
    logit_orig, attn_orig = raw_model(feat_tensor)
    prob_orig = torch.sigmoid(logit_orig).item()

print("-" * 50)
print(f"Raw Model Probability        : {prob_orig:.6f}")
print(f"Loaded TorchScript Probability: {prob_fresh:.6f}")
print(f"Absolute Difference          : {abs(prob_orig - prob_fresh):.2e}")
print("-" * 50)

assert np.isclose(prob_orig, prob_fresh, atol=1e-5), "Probabilities mismatch!"
assert np.allclose(attn_orig.numpy(), attn_fresh.numpy(), atol=1e-5), "Attention weights mismatch!"

# Test Explainability on fresh-loaded model
explain_dict = compute_time_aligned_attention(
    attention_weights=attn_fresh.squeeze().numpy(),
    num_frames_T=config_data["T"],
    downsample_factor=config_data["downsample_factor"],
    original_duration_sec=config_data["preprocess_config"]["segment_seconds"]
)
assert len(explain_dict["timestamps_sec"]) == 199
assert len(explain_dict["attention"]) == 199
print("FRESH-LOAD VERIFICATION PASSED: Outputs are bit-identical and explainability functions correctly!")


In [ ]:
# Cell 6: Create models/models_artifact.zip for transfer
zip_target = PROJECT_ROOT / "models" / "models_artifact"
shutil.make_archive(str(zip_target), "zip", str(ARTIFACT_DIR))
zip_file = PROJECT_ROOT / "models" / "models_artifact.zip"

print("=" * 60)
print(f"PRODUCTION ARTIFACT PACKAGE CREATED SUCCESSFULLY!")
print(f"Archive Path: {zip_file} ({zip_file.stat().st_size / 1e6:.1f} MB)")
print("=" * 60)

# Colab download trigger (if running in Colab)
try:
    from google.colab import files
    print("Triggering automatic browser download of models_artifact.zip...")
    files.download(str(zip_file))
except ImportError:
    print("Running locally: zip package is ready in models/models_artifact.zip")
